# Notebook 5 — RAG Knowledge Retrieval

## Explainable Apple Plant Disease Detection

### Objective

This notebook builds the **Retrieval-Augmented Generation (RAG) knowledge-retrieval layer** of the project.

The previous stages provide:

```text
ResNet50
   ↓
Disease prediction
   ↓
Confidence + uncertainty
   ↓
Grad-CAM explanation
   ↓
Trust / reliability assessment
```

RAG adds a separate knowledge layer:

```text
Predicted disease
       ↓
Semantic retrieval
       ↓
Relevant plant-disease knowledge
       ↓
Evidence with source metadata
```

**Important:** This notebook performs **retrieval only**. It does not generate an LLM answer yet. Notebook 6 will use the retrieved evidence to generate a human-readable explanation.

---

## Project Position

```text
Notebook 1 — Data Preparation                 ✅
Notebook 2 — Transfer Learning                ✅
Notebook 3 — Grad-CAM Explainability          ✅
Notebook 4 — Trust & Uncertainty              ✅
Notebook 5 — RAG Knowledge Retrieval           ← CURRENT
Notebook 6 — LLM Explanation
Notebook 7 — Human-in-the-Loop + Memory
Web Dashboard
```

## Why RAG?

The ResNet50 model is responsible for **visual classification**.

RAG is responsible for **retrieving supporting domain knowledge**.

For example:

```text
Image
 ↓
ResNet50
 ↓
Apple Scab
 ↓
RAG
 ↓
Symptoms + disease cycle + management information
 ↓
LLM
 ↓
Human-readable explanation
```

This separation is important because the LLM should not be treated as the image classifier. The classifier identifies the disease; RAG supplies traceable supporting information.

In [ ]:
# Install the retrieval dependencies.
# Run this cell once in Colab.

!pip -q install sentence-transformers faiss-cpu

In [ ]:
import os
import json
import re
import random
import numpy as np
import pandas as pd

from pathlib import Path
from urllib.parse import urlparse

import faiss
from sentence_transformers import SentenceTransformer

from google.colab import drive

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/XAI_Project")
RESULTS_DIR = PROJECT_DIR / "results"
RAG_DIR = RESULTS_DIR / "rag"

RAG_DIR.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_DIR)
print("RAG output directory:", RAG_DIR)

# 1. RAG Knowledge Scope

The classifier contains four Apple classes:

```text
0 → Apple Healthy
1 → Apple Scab
2 → Apple Black Rot
3 → Apple Cedar Apple Rust
```

The knowledge base focuses on these four classes.

For diseased classes, the knowledge records cover:

- Disease identity
- Causal organism
- Symptoms
- Disease cycle / spread
- Management principles
- Source information

For **Apple Healthy**, the knowledge base records that it is the project's healthy reference class and points to the same diagnostic resources used to distinguish disease symptoms.

The knowledge base is intentionally **source-traceable**. Every knowledge document contains its source organization and URL.

# 2. Source Selection

The initial knowledge base uses authoritative extension/university resources.

Primary sources include:

1. University of Minnesota Extension — Apple Scab
2. University of Minnesota Extension — Apple leaf disease diagnosis
3. University of Minnesota Extension — Cedar-Apple Rust
4. Penn State Extension — Apple disease control / fungicide timing
5. Penn State Extension — Apple scab disease review

These sources are used because the project needs traceable domain information rather than unsupported generated text.

The source URLs are stored with every chunk so the future dashboard/LLM can cite the evidence.

In [ ]:
SOURCES = {
    "umn_apple_scab": {
        "organization": "University of Minnesota Extension",
        "title": "Apple scab of apples and crabapples",
        "url": "https://extension.umn.edu/garden-and-home/yard-and-garden/gardening-in-minnesota/yard-and-garden-problems/apple-scab"
    },
    "umn_apple_leaf_diagnosis": {
        "organization": "University of Minnesota Extension",
        "title": "What's wrong with my plant? — Apples — Leaves — Spots on leaves",
        "url": "https://apps.extension.umn.edu/garden/diagnose/plant/fruit/apple/leavesspots.html"
    },
    "umn_cedar_rust": {
        "organization": "University of Minnesota Extension",
        "title": "Cedar-apple rust and related rust diseases",
        "url": "https://extension.umn.edu/garden-and-home/yard-and-garden/gardening-in-minnesota/yard-and-garden-problems/cedar-apple-rust"
    },
    "psu_disease_control": {
        "organization": "Penn State Extension",
        "title": "Apple Disease Control Toolbox — Fungicide Timing",
        "url": "https://extension.psu.edu/apple-disease-control-toolbox-fungicide-timing"
    },
    "psu_scab_review": {
        "organization": "Penn State Extension",
        "title": "Tree Fruit Disease — An Apple Scab Review",
        "url": "https://extension.psu.edu/tree-fruit-disease-an-apple-scab-review"
    }
}

with open(RAG_DIR / "sources.json", "w") as f:
    json.dump(SOURCES, f, indent=4)

print("Registered sources:", len(SOURCES))

# 3. Build a Source-Traceable Knowledge Base

The following records are **curated/paraphrased project knowledge** based on the listed sources.

We intentionally do not copy large passages from the sources. Instead, we store concise knowledge statements and preserve the source URL.

Each document has:

```text
document_id
disease
topic
text
source_organization
source_title
source_url
```

This metadata is preserved after chunking and embedding.

In [ ]:
KNOWLEDGE_DOCUMENTS = [
    {
        "document_id": "healthy_01",
        "disease": "Apple Healthy",
        "topic": "reference_class",
        "text": (
            "Apple Healthy is the reference class used by this image-classification project "
            "for leaves that do not show the disease patterns represented by the three "
            "disease classes in the dataset. Disease diagnosis resources describe characteristic "
            "visual signs such as spots, discoloration, lesions, and other disease symptoms; "
            "the healthy class is therefore used as the non-diseased reference category "
            "for this project."
        ),
        "source_organization": SOURCES["umn_apple_leaf_diagnosis"]["organization"],
        "source_title": SOURCES["umn_apple_leaf_diagnosis"]["title"],
        "source_url": SOURCES["umn_apple_leaf_diagnosis"]["url"],
    },

    {
        "document_id": "scab_01",
        "disease": "Apple Scab",
        "topic": "identity",
        "text": (
            "Apple scab is a fungal disease caused by Venturia inaequalis. It affects "
            "apple and crabapple and can affect both leaves and fruit."
        ),
        "source_organization": SOURCES["umn_apple_scab"]["organization"],
        "source_title": SOURCES["umn_apple_scab"]["title"],
        "source_url": SOURCES["umn_apple_scab"]["url"],
    },
    {
        "document_id": "scab_02",
        "disease": "Apple Scab",
        "topic": "symptoms",
        "text": (
            "Apple scab leaf spots are commonly olive-green to brown, may have a velvet-like "
            "or fringed appearance, and can become dark brown to black as they age. Heavily "
            "affected leaves can yellow and drop. Fruit lesions can become brown and corky, "
            "and infections early in fruit development can cause deformation and cracking."
        ),
        "source_organization": SOURCES["umn_apple_scab"]["organization"],
        "source_title": SOURCES["umn_apple_scab"]["title"],
        "source_url": SOURCES["umn_apple_scab"]["url"],
    },
    {
        "document_id": "scab_03",
        "disease": "Apple Scab",
        "topic": "disease_cycle",
        "text": (
            "Apple scab overwinters in infected fallen leaves. In spring, spores can be "
            "released and carried to developing leaves, flowers, fruit, and twigs. Moisture "
            "on plant surfaces supports infection, and secondary infection cycles can occur "
            "through the growing season when conditions are favorable."
        ),
        "source_organization": SOURCES["umn_apple_scab"]["organization"],
        "source_title": SOURCES["umn_apple_scab"]["title"],
        "source_url": SOURCES["umn_apple_scab"]["url"],
    },
    {
        "document_id": "scab_04",
        "disease": "Apple Scab",
        "topic": "management",
        "text": (
            "Management of apple scab includes cultural practices that improve airflow and "
            "help leaves dry quickly, removal of infected fallen leaves where appropriate, "
            "use of disease-resistant varieties, and properly timed disease-control programs. "
            "Fungicide timing is important when fungicides are used."
        ),
        "source_organization": SOURCES["psu_disease_control"]["organization"],
        "source_title": SOURCES["psu_disease_control"]["title"],
        "source_url": SOURCES["psu_disease_control"]["url"],
    },

    {
        "document_id": "blackrot_01",
        "disease": "Apple Black Rot",
        "topic": "identity",
        "text": (
            "Black rot of apple is associated with Botryosphaeria obtusa. It can affect "
            "leaves, fruit, and branches."
        ),
        "source_organization": SOURCES["umn_apple_leaf_diagnosis"]["organization"],
        "source_title": SOURCES["umn_apple_leaf_diagnosis"]["title"],
        "source_url": SOURCES["umn_apple_leaf_diagnosis"]["url"],
    },
    {
        "document_id": "blackrot_02",
        "disease": "Apple Black Rot",
        "topic": "symptoms",
        "text": (
            "Black rot can produce round leaf spots with a purple border and tan center. "
            "Fruit can develop large brown spots and rot while remaining relatively firm, "
            "and fruit can become mummified. Branch cankers may be sunken, reddish-brown, "
            "rough, or cracked. Leaves on girdled branches can wilt, die, and turn brown."
        ),
        "source_organization": SOURCES["umn_apple_leaf_diagnosis"]["organization"],
        "source_title": SOURCES["umn_apple_leaf_diagnosis"]["title"],
        "source_url": SOURCES["umn_apple_leaf_diagnosis"]["url"],
    },
    {
        "document_id": "blackrot_03",
        "disease": "Apple Black Rot",
        "topic": "management",
        "text": (
            "Apple disease management should consider disease timing, weather, tree growth, "
            "and orchard conditions. Penn State Extension provides apple disease-control "
            "timing information and emphasizes following current product labels and local "
            "production guidance when chemical controls are considered."
        ),
        "source_organization": SOURCES["psu_disease_control"]["organization"],
        "source_title": SOURCES["psu_disease_control"]["title"],
        "source_url": SOURCES["psu_disease_control"]["url"],
    },

    {
        "document_id": "rust_01",
        "disease": "Apple Cedar Apple Rust",
        "topic": "identity",
        "text": (
            "Cedar-apple rust is caused by Gymnosporangium juniperi-virginianae and requires "
            "hosts from two plant families to complete its life cycle. Apple and crabapple "
            "are important hosts in the Rosaceae family, while eastern red cedar and junipers "
            "can serve as alternate hosts."
        ),
        "source_organization": SOURCES["umn_cedar_rust"]["organization"],
        "source_title": SOURCES["umn_cedar_rust"]["title"],
        "source_url": SOURCES["umn_cedar_rust"]["url"],
    },
    {
        "document_id": "rust_02",
        "disease": "Apple Cedar Apple Rust",
        "topic": "symptoms",
        "text": (
            "On apple and crabapple, cedar-apple rust commonly produces leaf spots that are "
            "first yellow and later bright orange-red, often with a red border. Small raised "
            "black dots can occur in mature spots, and short finger-like fungal structures "
            "can develop on the underside of leaf spots."
        ),
        "source_organization": SOURCES["umn_cedar_rust"]["organization"],
        "source_title": SOURCES["umn_cedar_rust"]["title"],
        "source_url": SOURCES["umn_cedar_rust"]["url"],
    },
    {
        "document_id": "rust_03",
        "disease": "Apple Cedar Apple Rust",
        "topic": "spread",
        "text": (
            "Cedar-apple rust overwinters in galls on juniper or red cedar. In wet spring "
            "weather, these structures can release spores that are carried by wind to "
            "susceptible apple and other Rosaceae hosts. Wet leaves can support new infections."
        ),
        "source_organization": SOURCES["umn_cedar_rust"]["organization"],
        "source_title": SOURCES["umn_cedar_rust"]["title"],
        "source_url": SOURCES["umn_cedar_rust"]["url"],
    },
    {
        "document_id": "rust_04",
        "disease": "Apple Cedar Apple Rust",
        "topic": "management",
        "text": (
            "Management guidance for cedar-apple rust includes using resistant apple varieties "
            "where suitable, reducing nearby alternate hosts when appropriate, and pruning "
            "infected material where recommended. Extension guidance should be consulted for "
            "local management decisions."
        ),
        "source_organization": SOURCES["umn_cedar_rust"]["organization"],
        "source_title": SOURCES["umn_cedar_rust"]["title"],
        "source_url": SOURCES["umn_cedar_rust"]["url"],
    }
]

knowledge_df = pd.DataFrame(KNOWLEDGE_DOCUMENTS)

knowledge_path = RAG_DIR / "knowledge_documents.csv"
knowledge_df.to_csv(knowledge_path, index=False)

print("Knowledge documents:", len(knowledge_df))
print(knowledge_df.groupby("disease").size())
display(knowledge_df[["document_id", "disease", "topic"]])

# 4. Document Chunking

RAG works by retrieving small pieces of knowledge rather than entire large documents.

For this project the curated records are already short, so we use a lightweight word-based chunker.

Parameters:

```text
Chunk size  = 120 words
Overlap     = 30 words
```

The overlap helps preserve context when a longer record is split across chunks.

In [ ]:
CHUNK_SIZE_WORDS = 120
CHUNK_OVERLAP_WORDS = 30

def clean_text(text):
    text = re.sub(r"\s+", " ", str(text)).strip()
    return text

def chunk_text(text, chunk_size=120, overlap=30):
    words = clean_text(text).split()

    if len(words) <= chunk_size:
        return [clean_text(text)]

    chunks = []
    start = 0

    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunks.append(" ".join(words[start:end]))

        if end == len(words):
            break

        start = end - overlap

    return chunks

chunk_records = []

for _, row in knowledge_df.iterrows():
    chunks = chunk_text(
        row["text"],
        chunk_size=CHUNK_SIZE_WORDS,
        overlap=CHUNK_OVERLAP_WORDS
    )

    for chunk_id, chunk in enumerate(chunks):
        record = row.to_dict()
        record["chunk_id"] = chunk_id
        record["chunk_text"] = chunk
        chunk_records.append(record)

chunks_df = pd.DataFrame(chunk_records)

chunks_path = RAG_DIR / "knowledge_chunks.csv"
chunks_df.to_csv(chunks_path, index=False)

print("Total chunks:", len(chunks_df))
print(chunks_df.groupby("disease").size())

# 5. Embedding Model

Each knowledge chunk is converted into a numerical vector using a Sentence-Transformers model.

We use:

```text
sentence-transformers/all-MiniLM-L6-v2
```

This is a practical lightweight semantic embedding model for a project-scale RAG system.

The embedding dimension is determined directly from the loaded model rather than hard-coded.

In [ ]:
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

EMBEDDING_DIM = embedding_model.get_sentence_embedding_dimension()

print("Embedding model:", EMBEDDING_MODEL_NAME)
print("Embedding dimension:", EMBEDDING_DIM)

In [ ]:
# Create normalized embeddings.
# Normalization allows cosine similarity to be represented using inner product.

texts_for_embedding = chunks_df["chunk_text"].tolist()

embeddings = embedding_model.encode(
    texts_for_embedding,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

print("Embedding matrix shape:", embeddings.shape)

# 6. FAISS Vector Database

The embeddings are indexed with FAISS.

Because the embeddings are normalized, an inner-product FAISS index corresponds to cosine similarity.

The retrieval process becomes:

```text
User query
    ↓
Query embedding
    ↓
FAISS similarity search
    ↓
Top-k chunks
    ↓
Metadata + source URLs
```

This vector index is saved to Google Drive so it can be loaded by the future web application.

In [ ]:
index = faiss.IndexFlatIP(EMBEDDING_DIM)
index.add(embeddings)

INDEX_PATH = RAG_DIR / "apple_disease_faiss.index"
faiss.write_index(index, str(INDEX_PATH))

EMBEDDINGS_PATH = RAG_DIR / "knowledge_embeddings.npy"
np.save(EMBEDDINGS_PATH, embeddings)

print("FAISS index vectors:", index.ntotal)
print("Saved index:", INDEX_PATH)
print("Saved embeddings:", EMBEDDINGS_PATH)

In [ ]:
# Save embedding/index configuration.

rag_config = {
    "project": "Explainable Apple Plant Disease Detection",
    "knowledge_scope": [
        "Apple Healthy",
        "Apple Scab",
        "Apple Black Rot",
        "Apple Cedar Apple Rust"
    ],
    "embedding_model": EMBEDDING_MODEL_NAME,
    "embedding_dimension": EMBEDDING_DIM,
    "chunk_size_words": CHUNK_SIZE_WORDS,
    "chunk_overlap_words": CHUNK_OVERLAP_WORDS,
    "vector_index": "FAISS IndexFlatIP",
    "similarity": "cosine similarity via normalized inner product",
    "default_top_k": 5,
    "llm_used_in_this_notebook": False
}

with open(RAG_DIR / "rag_config.json", "w") as f:
    json.dump(rag_config, f, indent=4)

print("Saved:", RAG_DIR / "rag_config.json")

# 7. Retrieval Function

The central RAG function is:

```python
retrieve(query, disease=None, top_k=5)
```

It returns:

- Retrieved chunk
- Similarity score
- Disease
- Topic
- Source organization
- Source title
- Source URL

A disease filter can optionally restrict retrieval to the model's predicted disease.

This will be important in the final application because the ResNet50 prediction can be passed into RAG.

In [ ]:
def load_rag_resources():
    idx = faiss.read_index(str(INDEX_PATH))
    model = SentenceTransformer(EMBEDDING_MODEL_NAME)
    metadata = pd.read_csv(chunks_path)
    return idx, model, metadata

def retrieve(query, disease=None, top_k=5):
    query = clean_text(query)

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    # Retrieve more candidates when filtering by disease.
    search_k = min(max(top_k * 4, top_k), index.ntotal)

    scores, indices = index.search(query_embedding, search_k)

    results = []

    for score, idx in zip(scores[0], indices[0]):
        if idx < 0:
            continue

        row = chunks_df.iloc[int(idx)]

        if disease is not None:
            if str(row["disease"]).lower() != str(disease).lower():
                continue

        results.append({
            "rank": len(results) + 1,
            "similarity": float(score),
            "disease": row["disease"],
            "topic": row["topic"],
            "document_id": row["document_id"],
            "chunk_id": int(row["chunk_id"]),
            "text": row["chunk_text"],
            "source_organization": row["source_organization"],
            "source_title": row["source_title"],
            "source_url": row["source_url"]
        })

        if len(results) >= top_k:
            break

    return pd.DataFrame(results)

print("Retrieval function ready.")

# 8. Example Retrieval

A model prediction can be converted into a knowledge query.

For example:

```text
Prediction = Apple Scab
Trust = High
```

The query can ask for symptoms, disease information, and management.

The retriever returns source-traceable evidence.

No LLM is involved at this stage.

In [ ]:
example_query = (
    "Apple Scab symptoms, disease characteristics, disease cycle, "
    "and management information"
)

example_results = retrieve(
    example_query,
    disease="Apple Scab",
    top_k=5
)

display(
    example_results[
        [
            "rank",
            "similarity",
            "disease",
            "topic",
            "text",
            "source_organization",
            "source_url"
        ]
    ]
)

# 9. Retrieval Evaluation

A RAG system should be evaluated as a retrieval system before an LLM is connected.

We create representative queries for all four classes.

For each query, we define the expected disease.

The evaluation checks whether a chunk belonging to the expected disease appears within the top-k retrieved results.

We also calculate:

- Hit@1
- Hit@3
- Hit@5
- Mean Reciprocal Rank (MRR)

These are retrieval diagnostics, not disease-classification metrics.

In [ ]:
EVAL_QUERIES = [
    {
        "query": "What are the symptoms and appearance of apple scab on leaves?",
        "expected_disease": "Apple Scab"
    },
    {
        "query": "What causes apple scab and how does the disease spread?",
        "expected_disease": "Apple Scab"
    },
    {
        "query": "What are the symptoms of black rot on apple leaves?",
        "expected_disease": "Apple Black Rot"
    },
    {
        "query": "How is apple black rot managed?",
        "expected_disease": "Apple Black Rot"
    },
    {
        "query": "What does cedar apple rust look like on apple leaves?",
        "expected_disease": "Apple Cedar Apple Rust"
    },
    {
        "query": "How does cedar apple rust spread and what are management approaches?",
        "expected_disease": "Apple Cedar Apple Rust"
    },
    {
        "query": "What information describes the healthy reference class for apple leaves?",
        "expected_disease": "Apple Healthy"
    },
    {
        "query": "How can healthy apple leaves be distinguished from disease symptoms?",
        "expected_disease": "Apple Healthy"
    }
]

def evaluate_retrieval(eval_queries, k_values=(1, 3, 5)):
    rows = []

    reciprocal_ranks = []

    for item in eval_queries:
        results = retrieve(item["query"], top_k=max(k_values))
        expected = item["expected_disease"]

        disease_list = results["disease"].tolist()

        rr = 0.0
        for rank, disease in enumerate(disease_list, start=1):
            if disease == expected:
                rr = 1.0 / rank
                break

        reciprocal_ranks.append(rr)

        row = {
            "query": item["query"],
            "expected_disease": expected,
            "first_relevant_rank": int(1 / rr) if rr > 0 else None,
            "reciprocal_rank": rr
        }

        for k in k_values:
            row[f"hit_at_{k}"] = int(expected in disease_list[:k])

        rows.append(row)

    result_df = pd.DataFrame(rows)

    summary = {
        f"Hit@{k}": float(result_df[f"hit_at_{k}"].mean())
        for k in k_values
    }
    summary["MRR"] = float(np.mean(reciprocal_ranks))

    return result_df, summary

retrieval_eval_df, retrieval_summary = evaluate_retrieval(EVAL_QUERIES)

display(retrieval_eval_df)
print("Retrieval summary:")
for key, value in retrieval_summary.items():
    print(f"{key}: {value:.4f}")

In [ ]:
retrieval_eval_path = RAG_DIR / "retrieval_evaluation.csv"
retrieval_eval_df.to_csv(retrieval_eval_path, index=False)

with open(RAG_DIR / "retrieval_evaluation_summary.json", "w") as f:
    json.dump(retrieval_summary, f, indent=4)

print("Saved:", retrieval_eval_path)
print("Saved:", RAG_DIR / "retrieval_evaluation_summary.json")

# 10. Save Example Retrieval Results

The following cell stores representative retrieval outputs for later inspection and debugging.

This is useful when the web dashboard is developed because we can verify exactly what evidence the retriever returns for a disease query.

In [ ]:
example_queries = []

for disease in [
    "Apple Healthy",
    "Apple Scab",
    "Apple Black Rot",
    "Apple Cedar Apple Rust"
]:
    query = f"{disease} symptoms, identification, causes, and management"
    result = retrieve(query, disease=disease, top_k=5)

    for _, row in result.iterrows():
        example_queries.append({
            "query": query,
            "disease_filter": disease,
            "rank": row["rank"],
            "similarity": row["similarity"],
            "topic": row["topic"],
            "text": row["text"],
            "source_organization": row["source_organization"],
            "source_title": row["source_title"],
            "source_url": row["source_url"]
        })

example_retrieval_df = pd.DataFrame(example_queries)

example_retrieval_path = RAG_DIR / "example_retrieval_results.csv"
example_retrieval_df.to_csv(example_retrieval_path, index=False)

print("Saved:", example_retrieval_path)

# 11. Future Dashboard Interface

The web application will eventually call the same retrieval function.

Conceptually:

```python
prediction = "Apple Scab"

query = (
    f"{prediction} symptoms, causes, management, "
    "and prevention"
)

documents = retrieve(
    query,
    disease=prediction,
    top_k=5
)
```

The retrieved documents can then be passed to Notebook 6's LLM layer.

The LLM will receive **retrieved evidence + model output**, rather than being asked to independently diagnose the image.

In [ ]:
# Demonstration of the interface expected by the future dashboard.

def retrieve_for_prediction(predicted_class, trust_level=None, top_k=5):
    query = (
        f"{predicted_class}: symptoms, identification, disease characteristics, "
        f"causes, disease cycle, management, and prevention."
    )

    if trust_level is not None:
        query += f" Prediction trust level: {trust_level}."

    return retrieve(
        query=query,
        disease=predicted_class,
        top_k=top_k
    )

dashboard_demo = retrieve_for_prediction(
    predicted_class="Apple Cedar Apple Rust",
    trust_level="High",
    top_k=5
)

display(dashboard_demo)

# 12. Saved RAG Artifacts

The complete RAG layer is saved under:

```text
XAI_Project/
└── results/
    └── rag/
        ├── sources.json
        ├── knowledge_documents.csv
        ├── knowledge_chunks.csv
        ├── knowledge_embeddings.npy
        ├── apple_disease_faiss.index
        ├── rag_config.json
        ├── retrieval_evaluation.csv
        ├── retrieval_evaluation_summary.json
        └── example_retrieval_results.csv
```

These artifacts provide:

- Source-traceable knowledge
- Chunked knowledge
- Embeddings
- Persistent vector index
- Retrieval configuration
- Retrieval evaluation
- Example retrieval outputs

# Notebook 5 — Final Project Summary

## 1. What We Have Done

This notebook implemented the **Retrieval-Augmented Generation (RAG) retrieval layer** for the Apple plant disease project.

The system creates a source-traceable knowledge base for:

```text
Apple Healthy
Apple Scab
Apple Black Rot
Apple Cedar Apple Rust
```

The knowledge records contain information about disease identity, symptoms, disease cycle/spread, and management principles.

The knowledge was curated from authoritative extension/university resources and each record retains its source URL.

---

## 2. Input

The RAG system receives a **text query**.

The future web application will construct this query from the ResNet50 prediction.

For example:

```text
Predicted disease:
Apple Scab

Query:
Apple Scab symptoms, causes, disease cycle,
management, and prevention
```

The RAG embedding model converts the query into a vector.

---

## 3. Knowledge Representation

The knowledge records are divided into chunks.

Current configuration:

```text
Chunk size:
120 words

Chunk overlap:
30 words
```

Each chunk retains:

```text
Disease
Topic
Document ID
Chunk ID
Text
Source organization
Source title
Source URL
```

This makes the retrieved evidence traceable.

---

## 4. Embedding Model

The embedding model used is:

```text
sentence-transformers/all-MiniLM-L6-v2
```

The embedding dimension is determined automatically from the model.

Each knowledge chunk becomes a numerical vector:

```text
Knowledge text
      ↓
Sentence Transformer
      ↓
Embedding vector
```

The same embedding process is applied to user queries.

---

## 5. Vector Database

FAISS is used for vector similarity search.

The embeddings are normalized, so the FAISS inner-product similarity corresponds to cosine similarity.

The retrieval pipeline is:

```text
Query
  ↓
Query embedding
  ↓
FAISS similarity search
  ↓
Top-k knowledge chunks
  ↓
Source metadata
```

The default retrieval value is:

```text
Top-k = 5
```

---

## 6. Retrieval Output

The retrieval function returns:

```text
Rank
Similarity score
Disease
Topic
Document ID
Chunk ID
Retrieved text
Source organization
Source title
Source URL
```

This information will later be passed to the LLM.

---

## 7. Retrieval Evaluation

The RAG system was evaluated using representative queries for all four Apple classes.

The evaluation reports:

```text
Hit@1
Hit@3
Hit@5
MRR
```

These metrics measure whether the correct disease knowledge appears near the top of the retrieval results.

They are **retrieval metrics**, not image-classification metrics.

---

## 8. Why RAG Is Needed

The ResNet50 model answers:

> **What disease is visible in the image?**

Grad-CAM answers:

> **Which image regions contributed to the prediction?**

Trust & Uncertainty answers:

> **How reliable is the prediction?**

RAG answers:

> **What supporting domain knowledge is relevant to this prediction?**

Therefore:

```text
ResNet50
   ↓
Disease Prediction
   ↓
Trust + Grad-CAM
   ↓
RAG
   ↓
Evidence
   ↓
LLM
```

---

## 9. How RAG Helps the Next Stage

Notebook 6 will introduce the LLM.

The LLM will receive:

```text
Predicted disease
+
Confidence
+
Trust information
+
Grad-CAM information
+
Retrieved RAG evidence
```

The LLM can then generate a human-readable explanation grounded in the retrieved evidence.

The LLM should not replace the classifier.

Instead:

```text
ResNet50 → classification
RAG      → evidence retrieval
LLM      → explanation
```

---

## 10. How RAG Helps the Web Dashboard

The future dashboard can execute:

```text
User uploads Apple leaf
        ↓
ResNet50
        ↓
Disease prediction
        ↓
Trust / uncertainty
        ↓
Grad-CAM
        ↓
RAG retrieval
        ↓
Relevant knowledge
        ↓
LLM explanation
        ↓
Dashboard
```

The dashboard can show both the prediction and the evidence used to explain it.

The source URLs stored in the RAG metadata allow the application to provide traceable references to the underlying knowledge.

---

## 11. What We Have NOT Done Yet

This notebook deliberately does **not**:

- Generate LLM responses.
- Claim that RAG improves classification accuracy.
- Automatically change the ResNet50 prediction.
- Use Grad-CAM as localization ground truth.
- Use IoU or bounding boxes.
- Automatically retrain the classifier from retrieved knowledge.

Those functions belong to later stages.

---

## 12. Limitations

1. The current knowledge base is intentionally small and focused on the four project classes.
2. Retrieval quality depends on the quality and coverage of the knowledge base.
3. Semantic similarity does not guarantee factual correctness.
4. Source material can vary by geography, cultivar, weather, and production system.
5. Management recommendations should not be interpreted as universal prescriptions.
6. Chemical-control decisions require current local labels and applicable regulations.
7. The current retrieval evaluation uses a small manually defined query set and should be expanded for a stronger research evaluation.
8. The embedding model is a lightweight general-purpose model and can be replaced later if domain-specific retrieval requires it.

---

# 13. Final Architecture After Notebook 5

The project now contains:

```text
                 Apple Leaf Image
                        ↓
                   ResNet50
                        ↓
              Disease + Confidence
                        ↓
               Trust / Uncertainty
                        ↓
                   Grad-CAM
                        ↓
                  RAG Retrieval
                        ↓
              Source-Traceable Evidence
                        ↓
                     NEXT
                        ↓
               LLM Explanation
                        ↓
             Human-in-the-Loop
                        ↓
               Feedback / Memory
                        ↓
                 Web Dashboard
```

## Final Goal

The complete system is designed to answer five questions:

```text
WHAT?
→ What disease does the model predict?

HOW CONFIDENT?
→ How confident is the classifier?

WHY?
→ Which image regions influenced the prediction?

WHAT EVIDENCE?
→ What trusted knowledge supports the explanation?

WHAT IF UNCERTAIN?
→ Should the system request human review?
```

Notebook 5 therefore establishes the **knowledge-retrieval foundation** required for the next stage: **LLM-based grounded explanation**.